# DEH 30-Day PySpark Challenge
## Day 6 — withColumn, withColumnRenamed & Dropping Columns

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Step 4 — Load orders with explicit schema

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Rows: {orders_df.count()}")
orders_df.show(3)

## Step 5 — withColumn() — adding a new column

In [ ]:
# Add a new column
orders_df = orders_df.withColumn(
    "revenue",
    F.col("unit_price") * F.col("quantity")
)

orders_df.select("order_id", "unit_price", "quantity", "revenue").show(3)

## Step 6 — withColumn() — replacing an existing column

In [ ]:
# Replace an existing column — same name = overwrite
orders_df = orders_df.withColumn(
    "status",
    F.upper(F.col("status"))
)

orders_df.select("order_id", "status").show(3)

## Step 7 — Chaining withColumn() calls

In [ ]:
# Chain multiple withColumn calls
orders_df = orders_df \
    .withColumn("revenue", F.col("unit_price") * F.col("quantity")) \
    .withColumn("discounted_price", F.col("unit_price") * (1 - F.col("discount_pct") / 100)) \
    .withColumn("is_high_value", F.col("revenue") > 1000)

orders_df.select("order_id", "revenue", "discounted_price", "is_high_value").show(5)

## Step 8 — withColumnRenamed()

In [ ]:
# Rename a single column
orders_df = orders_df.withColumnRenamed("unit_price", "price")
orders_df.select("order_id", "price").show(3)

In [ ]:
# Chain multiple renames
orders_df = orders_df \
    .withColumnRenamed("customer_id", "cust_id") \
    .withColumnRenamed("product_id", "prod_id") \
    .withColumnRenamed("order_date", "date")

print(orders_df.columns)

In [ ]:
# Rename all columns at once using toDF()
small_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv") \
    .select("order_id", "customer_id", "unit_price")

renamed_df = small_df.toDF("id", "cust", "price")
renamed_df.show(3)

## Step 9 — drop()

In [ ]:
# Reload clean orders
orders_clean = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Before drop: {orders_clean.columns}")

# Drop a single column
orders_clean = orders_clean.drop("discount_pct")
print(f"After dropping discount_pct: {len(orders_clean.columns)} columns")

# Drop multiple columns at once
orders_clean = orders_clean.drop("payment_method", "region")
print(f"After dropping payment_method & region: {len(orders_clean.columns)} columns")
print(orders_clean.columns)

In [ ]:
# Drop a column that does not exist — no error thrown
result = orders_clean.drop("nonexistent_column")
print(f"Drop nonexistent column — no error, columns unchanged: {len(result.columns)}")

---
## Assignment — Day 6

---

### Task 1
From `orders.csv`, use `withColumn()` to add three new columns:
- `revenue` = `unit_price * quantity`
- `discounted_price` = `unit_price * (1 - discount_pct / 100)`
- `is_delivered` = boolean, true when status equals 'Delivered'

Show all three new columns alongside `order_id`.

In [ ]:
# Task 1 — Write your code here


### Task 2
Use `withColumnRenamed()` to rename `unit_price` to `price` and `customer_id` to `cust_id`.  
Print the column list to verify.

In [ ]:
# Task 2 — Write your code here


### Task 3
From `customers.csv`, drop the columns `email`, `country`, and `signup_date`.  
How many columns remain? Print them.

In [ ]:
# Task 3 — Write your code here


### Task 4
Using `orders.csv`, add a `revenue` column, then try dropping a column that does not exist — like `nonexistent_col`.  
What happens? Does it throw an error?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*